<a href="https://colab.research.google.com/github/SanviBallal/Factory-Shipping-Optimization/blob/main/06_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import numpy as np
import os

required_files = [
    "Nassau Candy Distributor.csv",
    "combined_product_factory_mapping.csv",
    "customer_geographical_data.csv",
    "factory_customer_distances.csv",
    "nearest_factory_assignments.csv"
]

print("DAY 6 - FILE VALIDATION")
print("-" * 40)

for file in required_files:
    if os.path.exists("/content/" + file):
        print("FOUND:", file)
    else:
        print("MISSING:", file)


DAY 6 - FILE VALIDATION
----------------------------------------
FOUND: Nassau Candy Distributor.csv
FOUND: combined_product_factory_mapping.csv
FOUND: customer_geographical_data.csv
FOUND: factory_customer_distances.csv
FOUND: nearest_factory_assignments.csv


In [2]:


df = pd.read_csv("/content/Nassau Candy Distributor.csv")

factory_mapping_df = pd.read_csv(
    "/content/combined_product_factory_mapping.csv"
)

customer_geo_df = pd.read_csv(
    "/content/customer_geographical_data.csv"
)

distance_df = pd.read_csv(
    "/content/factory_customer_distances.csv"
)

nearest_factory_df = pd.read_csv(
    "/content/nearest_factory_assignments.csv"
)

# Display dataset dimensions

print("ORIGINAL TRANSACTION DATA:", df.shape)
print("PRODUCT-FACTORY MAPPING:", factory_mapping_df.shape)
print("CUSTOMER GEOGRAPHICAL DATA:", customer_geo_df.shape)
print("FACTORY-CUSTOMER DISTANCES:", distance_df.shape)
print("NEAREST FACTORY ASSIGNMENTS:", nearest_factory_df.shape)


ORIGINAL TRANSACTION DATA: (10194, 18)
PRODUCT-FACTORY MAPPING: (15, 4)
CUSTOMER GEOGRAPHICAL DATA: (655, 9)
FACTORY-CUSTOMER DISTANCES: (3275, 13)
NEAREST FACTORY ASSIGNMENTS: (655, 8)


In [3]:


datasets = {
    "Original Data": df,
    "Factory Mapping": factory_mapping_df,
    "Customer Geography": customer_geo_df,
    "Factory-Customer Distances": distance_df,
    "Nearest Factory": nearest_factory_df
}

for name, data in datasets.items():
    print("\n", "=" * 60)
    print(name.upper())
    print("=" * 60)
    print(data.columns.tolist())



ORIGINAL DATA
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name', 'Sales', 'Units', 'Gross Profit', 'Cost']

FACTORY MAPPING
['Product Name', 'Current Factory', 'Latitude', 'Longitude']

CUSTOMER GEOGRAPHY
['Country/Region', 'State/Province', 'City', 'Postal Code', 'Latitude', 'Longitude', 'Accuracy', 'Coordinate Source', 'Customer Location ID']

FACTORY-CUSTOMER DISTANCES
['Factory', 'Factory Latitude', 'Factory Longitude', 'Country/Region', 'State/Province', 'City', 'Postal Code', 'Latitude', 'Longitude', 'Accuracy', 'Coordinate Source', 'Customer Location ID', 'Distance (km)']

NEAREST FACTORY
['Customer Location ID', 'Country/Region', 'State/Province', 'City', 'Postal Code', 'Factory', 'Distance (km)', 'Coordinate Source']


In [4]:


print("CUSTOMER LOCATION KEY CHECK")
print("-" * 45)

customer_keys = [
    "Country/Region",
    "State/Province",
    "City",
    "Postal Code"
]

for col in customer_keys:
    print(
        col,
        "| Original:", df[col].dtype,
        "| Geography:", customer_geo_df[col].dtype
    )

print("\nPRODUCT MAPPING CHECK")
print("-" * 45)

print("Unique products in original data:",
      df["Product Name"].nunique())

print("Unique products in factory mapping:",
      factory_mapping_df["Product Name"].nunique())

print("Duplicate products in factory mapping:",
      factory_mapping_df["Product Name"].duplicated().sum())

print("\nCUSTOMER GEOGRAPHY CHECK")
print("-" * 45)

print("Duplicate customer location IDs:",
      customer_geo_df["Customer Location ID"].duplicated().sum())

print("Duplicate customer location combinations:",
      customer_geo_df.duplicated(subset=customer_keys).sum())


CUSTOMER LOCATION KEY CHECK
---------------------------------------------
Country/Region | Original: object | Geography: object
State/Province | Original: object | Geography: object
City | Original: object | Geography: object
Postal Code | Original: object | Geography: object

PRODUCT MAPPING CHECK
---------------------------------------------
Unique products in original data: 15
Unique products in factory mapping: 15
Duplicate products in factory mapping: 0

CUSTOMER GEOGRAPHY CHECK
---------------------------------------------
Duplicate customer location IDs: 0
Duplicate customer location combinations: 0


In [5]:


customer_keys = [
    "Country/Region",
    "State/Province",
    "City",
    "Postal Code"
]

# Create temporary standardized matching columns
for col in customer_keys:
    df[col + "_key"] = (
        df[col].astype("string").str.strip().str.upper()
    )

    customer_geo_df[col + "_key"] = (
        customer_geo_df[col].astype("string").str.strip().str.upper()
    )

matching_keys = [col + "_key" for col in customer_keys]

# Get unique transaction locations
transaction_locations = df[matching_keys].drop_duplicates()

# Get unique geographical locations
geographical_locations = customer_geo_df[matching_keys].drop_duplicates()

# Check matching locations
location_check = transaction_locations.merge(
    geographical_locations,
    on=matching_keys,
    how="left",
    indicator=True
)

print("CUSTOMER LOCATION MATCHING REPORT")
print("-" * 45)

print("Unique transaction locations:",
      len(transaction_locations))

print("Matched locations:",
      (location_check["_merge"] == "both").sum())

print("Unmatched locations:",
      (location_check["_merge"] == "left_only").sum())

if (location_check["_merge"] == "left_only").any():
    print("\nUNMATCHED LOCATION SAMPLES:")
    print(
        location_check[
            location_check["_merge"] == "left_only"
        ][matching_keys].head(10)
    )


CUSTOMER LOCATION MATCHING REPORT
---------------------------------------------
Unique transaction locations: 655
Matched locations: 603
Unmatched locations: 52

UNMATCHED LOCATION SAMPLES:
    Country/Region_key State/Province_key    City_key Postal Code_key
39       UNITED STATES         NEW JERSEY      LINDEN            7036
101      UNITED STATES      MASSACHUSETTS      REVERE            2151
135      UNITED STATES        CONNECTICUT  MIDDLETOWN            6457
139      UNITED STATES      MASSACHUSETTS    LAWRENCE            1841
145      UNITED STATES      MASSACHUSETTS    FRANKLIN            2038
153      UNITED STATES      NEW HAMPSHIRE       DOVER            3820
164      UNITED STATES         NEW JERSEY    LAKEWOOD            8701
177      UNITED STATES        CONNECTICUT     MERIDEN            6450
189      UNITED STATES         NEW JERSEY  PLAINFIELD            7060
194      UNITED STATES         NEW JERSEY  HACKENSACK            7601


In [6]:


unmatched_locations = location_check[
    location_check["_merge"] == "left_only"
].copy()

# Display original values and possible geographical matches
for _, row in unmatched_locations.head(10).iterrows():

    print("\n" + "-" * 55)
    print("CITY:", row["City_key"])
    print("STATE:", row["State/Province_key"])
    print("ORIGINAL POSTAL CODE:", row["Postal Code_key"])

    candidates = customer_geo_df[
        (customer_geo_df["City_key"] == row["City_key"]) &
        (customer_geo_df["State/Province_key"] == row["State/Province_key"]) &
        (customer_geo_df["Country/Region_key"] == row["Country/Region_key"])
    ]

    print("GEOGRAPHICAL DATA POSTAL CODES:",
          candidates["Postal Code"].tolist()[:10])



-------------------------------------------------------
CITY: LINDEN
STATE: NEW JERSEY
ORIGINAL POSTAL CODE: 7036
GEOGRAPHICAL DATA POSTAL CODES: ['07036']

-------------------------------------------------------
CITY: REVERE
STATE: MASSACHUSETTS
ORIGINAL POSTAL CODE: 2151
GEOGRAPHICAL DATA POSTAL CODES: ['02151']

-------------------------------------------------------
CITY: MIDDLETOWN
STATE: CONNECTICUT
ORIGINAL POSTAL CODE: 6457
GEOGRAPHICAL DATA POSTAL CODES: ['06457']

-------------------------------------------------------
CITY: LAWRENCE
STATE: MASSACHUSETTS
ORIGINAL POSTAL CODE: 1841
GEOGRAPHICAL DATA POSTAL CODES: ['01841']

-------------------------------------------------------
CITY: FRANKLIN
STATE: MASSACHUSETTS
ORIGINAL POSTAL CODE: 2038
GEOGRAPHICAL DATA POSTAL CODES: ['02038']

-------------------------------------------------------
CITY: DOVER
STATE: NEW HAMPSHIRE
ORIGINAL POSTAL CODE: 3820
GEOGRAPHICAL DATA POSTAL CODES: ['03820']

-------------------------------------

In [7]:


def standardize_postal_code(country, postal_code):
    postal_code = str(postal_code).strip().upper()
    postal_code = postal_code.replace(" ", "")
    postal_code = postal_code.replace(".0", "")

    if country == "UNITED STATES":
        return postal_code.zfill(5)

    return postal_code


# Standardize transaction postal codes
df["Postal Code_key"] = df.apply(
    lambda row: standardize_postal_code(
        row["Country/Region_key"],
        row["Postal Code"]
    ),
    axis=1
)

# Standardize geographical postal codes
customer_geo_df["Postal Code_key"] = customer_geo_df.apply(
    lambda row: standardize_postal_code(
        row["Country/Region_key"],
        row["Postal Code"]
    ),
    axis=1
)

# Recheck location matching
transaction_locations = df[matching_keys].drop_duplicates()
geographical_locations = customer_geo_df[matching_keys].drop_duplicates()

location_check = transaction_locations.merge(
    geographical_locations,
    on=matching_keys,
    how="left",
    indicator=True
)

print("UPDATED CUSTOMER LOCATION MATCHING REPORT")
print("-" * 50)

print("Unique transaction locations:", len(transaction_locations))
print("Matched locations:",
      (location_check["_merge"] == "both").sum())
print("Unmatched locations:",
      (location_check["_merge"] == "left_only").sum())

if (location_check["_merge"] == "left_only").any():
    print("\nREMAINING UNMATCHED LOCATIONS:")
    print(
        location_check[
            location_check["_merge"] == "left_only"
        ][matching_keys].head(15).to_string(index=False)
    )


UPDATED CUSTOMER LOCATION MATCHING REPORT
--------------------------------------------------
Unique transaction locations: 655
Matched locations: 655
Unmatched locations: 0


In [8]:


geo_lookup = customer_geo_df[
    matching_keys + [
        "Latitude",
        "Longitude",
        "Accuracy",
        "Coordinate Source",
        "Customer Location ID"
    ]
].copy()

# Merge geographical information into transactions
df = df.merge(
    geo_lookup,
    on=matching_keys,
    how="left",
    validate="many_to_one",
    indicator=True
)

print("CUSTOMER GEOGRAPHICAL MERGE REPORT")
print("-" * 50)

print("Merged dataset shape:", df.shape)
print("Original transaction records:", 10194)
print("Records after merge:", len(df))

print("Matched transaction records:",
      (df["_merge"] == "both").sum())

print("Unmatched transaction records:",
      (df["_merge"] == "left_only").sum())

print("Missing customer latitudes:",
      df["Latitude"].isna().sum())

print("Missing customer longitudes:",
      df["Longitude"].isna().sum())

# Remove merge indicator and temporary matching columns
df.drop(columns=["_merge"] + matching_keys, inplace=True)

print("\nFINAL MERGED DATASET SHAPE:", df.shape)


CUSTOMER GEOGRAPHICAL MERGE REPORT
--------------------------------------------------
Merged dataset shape: (10194, 28)
Original transaction records: 10194
Records after merge: 10194
Matched transaction records: 10194
Unmatched transaction records: 0
Missing customer latitudes: 0
Missing customer longitudes: 0

FINAL MERGED DATASET SHAPE: (10194, 23)


In [9]:


df = df.merge(
    factory_mapping_df[
        ["Product Name", "Current Factory"]
    ],
    on="Product Name",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("PRODUCT-FACTORY MERGE REPORT")
print("-" * 45)

print("Dataset shape:", df.shape)
print("Matched transactions:",
      (df["_merge"] == "both").sum())
print("Unmatched transactions:",
      (df["_merge"] == "left_only").sum())

print("\nFACTORY-WISE TRANSACTION COUNT")
print(
    df["Current Factory"].value_counts(dropna=False)
)

# Remove merge indicator
df.drop(columns=["_merge"], inplace=True)


PRODUCT-FACTORY MERGE REPORT
---------------------------------------------
Dataset shape: (10194, 25)
Matched transactions: 10194
Unmatched transactions: 0

FACTORY-WISE TRANSACTION COUNT
Current Factory
Lot's O' Nuts        5692
Wicked Choccy's      4152
Secret Factory        217
The Other Factory     100
Sugar Shack            33
Name: count, dtype: int64


In [10]:


nearest_lookup = nearest_factory_df[
    [
        "Customer Location ID",
        "Factory",
        "Distance (km)"
    ]
].rename(columns={
    "Factory": "Nearest Factory",
    "Distance (km)": "Nearest Factory Distance (km)"
})

df = df.merge(
    nearest_lookup,
    on="Customer Location ID",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("NEAREST FACTORY MERGE REPORT")
print("-" * 45)

print("Dataset shape:", df.shape)

print("Matched transactions:",
      (df["_merge"] == "both").sum())

print("Unmatched transactions:",
      (df["_merge"] == "left_only").sum())

print("Missing nearest factories:",
      df["Nearest Factory"].isna().sum())

print("Missing nearest distances:",
      df["Nearest Factory Distance (km)"].isna().sum())

df.drop(columns=["_merge"], inplace=True)


NEAREST FACTORY MERGE REPORT
---------------------------------------------
Dataset shape: (10194, 27)
Matched transactions: 10194
Unmatched transactions: 0
Missing nearest factories: 0
Missing nearest distances: 0


In [11]:


current_distance_lookup = distance_df[
    ["Factory", "Customer Location ID", "Distance (km)"]
].rename(columns={
    "Factory": "Current Factory",
    "Distance (km)": "Current Factory Distance (km)"
})

df = df.merge(
    current_distance_lookup,
    on=["Current Factory", "Customer Location ID"],
    how="left",
    validate="many_to_one",
    indicator=True
)

print("CURRENT FACTORY DISTANCE REPORT")
print("-" * 50)

print("Dataset shape:", df.shape)

print("Matched transactions:",
      (df["_merge"] == "both").sum())

print("Unmatched transactions:",
      (df["_merge"] == "left_only").sum())

print("Missing current factory distances:",
      df["Current Factory Distance (km)"].isna().sum())

df.drop(columns=["_merge"], inplace=True)


CURRENT FACTORY DISTANCE REPORT
--------------------------------------------------
Dataset shape: (10194, 28)
Matched transactions: 10194
Unmatched transactions: 0
Missing current factory distances: 0


In [12]:


# Calculate potential distance reduction
df["Potential Distance Reduction (km)"] = (
    df["Current Factory Distance (km)"]
    - df["Nearest Factory Distance (km)"]
)

# Calculate percentage distance reduction
df["Potential Distance Reduction (%)"] = np.where(
    df["Current Factory Distance (km)"] > 0,
    (
        df["Potential Distance Reduction (km)"]
        / df["Current Factory Distance (km)"]
    ) * 100,
    0
)

# Check whether current factory is already the nearest
df["Current Factory Is Nearest"] = (
    df["Current Factory"] == df["Nearest Factory"]
)

print("DISTANCE COMPARISON REPORT")
print("-" * 50)

print("Average current factory distance:",
      round(df["Current Factory Distance (km)"].mean(), 2), "km")

print("Average nearest factory distance:",
      round(df["Nearest Factory Distance (km)"].mean(), 2), "km")

print("Average potential distance reduction:",
      round(df["Potential Distance Reduction (km)"].mean(), 2), "km")

print("Average potential reduction percentage:",
      round(df["Potential Distance Reduction (%)"].mean(), 2), "%")

print("\nCURRENT FACTORY ALREADY NEAREST")
print(df["Current Factory Is Nearest"].value_counts())

print("\nNEGATIVE DISTANCE REDUCTIONS:",
      (df["Potential Distance Reduction (km)"] < -0.001).sum())

print("\nFEATURE PREVIEW")
print(
    df[
        [
            "Product Name",
            "Current Factory",
            "Nearest Factory",
            "Current Factory Distance (km)",
            "Nearest Factory Distance (km)",
            "Potential Distance Reduction (km)",
            "Potential Distance Reduction (%)",
            "Current Factory Is Nearest"
        ]
    ].head(10)
)


DISTANCE COMPARISON REPORT
--------------------------------------------------
Average current factory distance: 2009.06 km
Average nearest factory distance: 818.74 km
Average potential distance reduction: 1190.32 km
Average potential reduction percentage: 45.4 %

CURRENT FACTORY ALREADY NEAREST
Current Factory Is Nearest
False    6967
True     3227
Name: count, dtype: int64

NEGATIVE DISTANCE REDUCTIONS: 0

FEATURE PREVIEW
                        Product Name  Current Factory    Nearest Factory  \
0         Wonka Bar - Milk Chocolate  Wicked Choccy's  The Other Factory   
1  Wonka Bar - Triple Dazzle Caramel  Wicked Choccy's     Secret Factory   
2  Wonka Bar - Nutty Crunch Surprise    Lot's O' Nuts     Secret Factory   
3     Wonka Bar -Scrumdiddlyumptious    Lot's O' Nuts     Secret Factory   
4  Wonka Bar - Triple Dazzle Caramel  Wicked Choccy's    Wicked Choccy's   
5     Wonka Bar -Scrumdiddlyumptious    Lot's O' Nuts  The Other Factory   
6  Wonka Bar - Triple Dazzle Caramel  Wic

In [13]:

# DAY 6 - FINANCIAL AND DATE FEATURES

# Financial features
df["Profit Margin (%)"] = np.where(
    df["Sales"] > 0,
    (df["Gross Profit"] / df["Sales"]) * 100,
    0
)

df["Sales per Unit"] = df["Sales"] / df["Units"]
df["Cost per Unit"] = df["Cost"] / df["Units"]
df["Profit per Unit"] = df["Gross Profit"] / df["Units"]

# Date features
df["Order Date"] = pd.to_datetime(
    df["Order Date"],
    dayfirst=True,
    errors="coerce"
)

df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.month
df["Order Month Name"] = df["Order Date"].dt.month_name()
df["Order Quarter"] = df["Order Date"].dt.quarter
df["Order Day of Week"] = df["Order Date"].dt.day_name()

# Display results
print("FINANCIAL AND DATE FEATURE REPORT")
print("-" * 50)

new_features = [
    "Profit Margin (%)",
    "Sales per Unit",
    "Cost per Unit",
    "Profit per Unit",
    "Order Year",
    "Order Month",
    "Order Month Name",
    "Order Quarter",
    "Order Day of Week"
]

print("New features created:", len(new_features))
print("\nFeature preview:")

print(df[new_features].head(10))

print("\nMissing values in new features:")
print(df[new_features].isnull().sum())


FINANCIAL AND DATE FEATURE REPORT
--------------------------------------------------
New features created: 9

Feature preview:
   Profit Margin (%)  Sales per Unit  Cost per Unit  Profit per Unit  \
0          64.923077            3.25           1.14             2.11   
1          65.333333            3.75           1.30             2.45   
2          71.346705            3.49           1.00             2.49   
3          69.444444            3.60           1.10             2.50   
4          65.333333            3.75           1.30             2.45   
5          69.444444            3.60           1.10             2.50   
6          65.333333            3.75           1.30             2.45   
7          64.923077            3.25           1.14             2.11   
8          71.346705            3.49           1.00             2.49   
9          64.923077            3.25           1.14             2.11   

   Order Year  Order Month Order Month Name  Order Quarter Order Day of Week  
0

In [14]:

# DAY 6 - FINAL FEATURE VALIDATION

print("FINAL FEATURE ENGINEERING VALIDATION")
print("=" * 55)

print("Final dataset shape:", df.shape)
print("Original transaction count:", 10194)
print("Final transaction count:", len(df))

print("\nDuplicate Row IDs:", df["Row ID"].duplicated().sum())

print("\nMissing values in entire dataset:",
      df.isnull().sum().sum())

print("\nDistance validation:")
print(
    "Negative distance reductions:",
    (df["Potential Distance Reduction (km)"] < -0.001).sum()
)

print(
    "Current factory already nearest:",
    df["Current Factory Is Nearest"].sum()
)

print("\nFinancial validation:")
print("Minimum profit margin:",
      round(df["Profit Margin (%)"].min(), 2), "%")

print("Maximum profit margin:",
      round(df["Profit Margin (%)"].max(), 2), "%")

print("\nOrder year distribution:")
print(df["Order Year"].value_counts().sort_index())

print("\nFinal validation completed.")


FINAL FEATURE ENGINEERING VALIDATION
Final dataset shape: (10194, 39)
Original transaction count: 10194
Final transaction count: 10194

Duplicate Row IDs: 0

Missing values in entire dataset: 200

Distance validation:
Negative distance reductions: 0
Current factory already nearest: 3227

Financial validation:
Minimum profit margin: 7.69 %
Maximum profit margin: 80.0 %

Order year distribution:
Order Year
2024    4181
2025    6013
Name: count, dtype: int64

Final validation completed.


In [15]:

# DAY 6 - MISSING VALUE INVESTIGATION

missing_report = df.isnull().sum()

missing_report = missing_report[missing_report > 0]

print("COLUMNS CONTAINING MISSING VALUES")
print("-" * 45)

print(missing_report)

print("\nTOTAL MISSING VALUES:",
      missing_report.sum())

print("\nMISSING VALUE PERCENTAGE:")
print((missing_report / len(df) * 100).round(2))

print("\nCHECK IMPORTANT FEATURES:")
important_columns = [
    "Latitude",
    "Longitude",
    "Current Factory",
    "Nearest Factory",
    "Current Factory Distance (km)",
    "Nearest Factory Distance (km)",
    "Potential Distance Reduction (km)",
    "Profit Margin (%)",
    "Order Year"
]

print(df[important_columns].isnull().sum())


COLUMNS CONTAINING MISSING VALUES
---------------------------------------------
Accuracy    200
dtype: int64

TOTAL MISSING VALUES: 200

MISSING VALUE PERCENTAGE:
Accuracy    1.96
dtype: float64

CHECK IMPORTANT FEATURES:
Latitude                             0
Longitude                            0
Current Factory                      0
Nearest Factory                      0
Current Factory Distance (km)        0
Nearest Factory Distance (km)        0
Potential Distance Reduction (km)    0
Profit Margin (%)                    0
Order Year                           0
dtype: int64


In [16]:

# DAY 6 - SAVE FEATURE ENGINEERED DATASET

output_file = "/content/feature_engineered_dataset.csv"

df.to_csv(output_file, index=False)

print("Feature-engineered dataset saved successfully!")
print("File:", output_file)
print("Dataset shape:", df.shape)
print("Total transactions:", len(df))
print("Total features:", len(df.columns))


Feature-engineered dataset saved successfully!
File: /content/feature_engineered_dataset.csv
Dataset shape: (10194, 39)
Total transactions: 10194
Total features: 39


# Day 6: Feature Engineering – Findings and Conclusion

## 1. Objective
The objective of Day 6 was to combine the original transaction dataset with factory mapping, customer geographical information and factory distance datasets to create a feature-engineered dataset for further analysis and optimization.

## 2. Dataset Integration
The following datasets were successfully integrated:

- Nassau Candy Distributor transaction dataset
- Product-to-factory mapping
- Customer geographical data
- Factory-to-customer distance data
- Nearest factory assignments

All 10,194 transactions were successfully matched with customer geographical information, current factory assignments, nearest factory assignments and their respective distances.

## 3. Distance-Based Feature Engineering
The following features were created:

- Current Factory Distance (km)
- Nearest Factory Distance (km)
- Potential Distance Reduction (km)
- Potential Distance Reduction (%)
- Current Factory Is Nearest

### Key Findings
- Average current factory distance: approximately 2,009.06 km.
- Average nearest factory distance: approximately 818.74 km.
- Average potential distance reduction: approximately 1,190.32 km.
- Average potential distance reduction percentage: approximately 45.4%.
- Current factory was already the nearest factory for 3,227 transactions.
- For 6,967 transactions, another factory was geographically closer.

These results indicate potential opportunities for improving factory allocation based on geographical distance.

**Note:** Distances are straight-line Haversine estimates, not actual road distances, transportation costs or delivery times.

## 4. Financial Feature Engineering
The following features were created:

- Profit Margin (%)
- Sales per Unit
- Cost per Unit
- Profit per Unit

The calculated profit margin ranged from 7.69% to 80.00%.

These features will help evaluate financial performance while comparing alternative factory allocation scenarios.

## 5. Date Feature Engineering
The following features were extracted from Order Date:

- Order Year
- Order Month
- Order Month Name
- Order Quarter
- Order Day of Week

Year-wise transaction distribution:

| Year | Transactions |
|---|---:|
| 2024 | 4,181 |
| 2025 | 6,013 |

Ship Date was excluded from feature engineering because the earlier investigation identified suspicious shipping-date values.

## 6. Final Validation
- Original transactions: 10,194
- Final transactions: 10,194
- Final dataset columns: 39
- Duplicate Row IDs: 0
- Missing values in critical features: 0
- Negative distance reductions: 0
- Missing Accuracy values: 200

The missing Accuracy values were retained without artificial replacement because geographical coordinates and distance calculations were available.

## 7. Output
The final feature-engineered dataset was saved as:

`feature_engineered_dataset.csv`

Dataset shape: **10,194 rows × 39 columns**

## 8. Conclusion
Day 6 successfully integrated the transaction, factory mapping, customer location and distance datasets and created geographical, financial and date-based features.

The results reveal that some products are currently assigned to factories that are not geographically closest to their customers. This creates opportunities for further factory allocation analysis.

The engineered dataset will serve as the foundation for subsequent analysis, scenario simulation and factory reallocation recommendations.

**Day 6: Successfully completed.**